In [1]:
import random
import numpy as np
import torch
import torch.nn as nn

its incomplete.

In [2]:
# create embeddings. 

dic = {
    "<PAD>" : 0,
    "<SOS>" : 1,
    "<EOS>" : 2,
}
original = set()
idx = 3

with open("output.txt", "r") as f:
    for token in f.read().split():
        if token not in original:
            original.add(token)
            dic[token] = idx
            idx += 1

NUM_UNIQUE_WORDS = len(dic)

In [4]:
# converting the dataset of words into indices

dataset = []

MAX_SEQ_LEN = 50 # Set a reasonable limit

with open("output.txt", "r") as f:
    for line in f.readlines():
        sentence = [dic["<SOS>"]]
        # Truncate the split list if it's too long
        words = str(line).split()[:MAX_SEQ_LEN] 
        for word in words:
            sentence.append(dic.get(word, dic.get("<PAD>", 0))) # Handle unknown words safely
        
        sentence.append(dic["<EOS>"])
        dataset.append(sentence)

In [5]:
class GRU(nn.Module): # unidirectional
    def __init__(self, input_size, hidden_size) -> None:
        super(GRU, self).__init__()

        self.sigmoid = nn.Sigmoid()
        self.tanh = nn.Tanh()

        self.restore_fc = nn.Linear(input_size + hidden_size, hidden_size)
        self.update_fc = nn.Linear(input_size + hidden_size, hidden_size)

        self.pseudo_current_hidden_fc = nn.Linear(input_size + hidden_size, hidden_size)

    def forward(self, x, h_prev):

        gate_input = torch.concat([x, h_prev], dim=-1)
        
        restore_gate = self.sigmoid(self.restore_fc(gate_input))
        update_gate = self.sigmoid(self.update_fc(gate_input))

        psuedo_hidden_input = torch.concat([x, restore_gate * h_prev], dim=-1)
        psuedo_hidden = self.tanh(self.pseudo_current_hidden_fc(psuedo_hidden_input))

        h_t = (1 - update_gate) * h_prev + update_gate * psuedo_hidden

        return h_t

In [6]:
class Encoder(nn.Module):
    def __init__(self, hidden_size, embedding, layers=1, dropout=0) -> None:
        super(Encoder, self).__init__()

        self.hidden_size = hidden_size
        self.embedding = embedding
        
        self.gru = nn.GRU(input_size=hidden_size, hidden_size=hidden_size, num_layers=layers, dropout=(0 if layers == 1 else dropout), bidirectional=True, batch_first=True)
    
    def forward(self, input_seq, input_lens, hidden=None):
        
        embedded = self.embedding(input_seq)

        packed = nn.utils.rnn.pack_padded_sequence(embedded, input_lens, batch_first=True, enforce_sorted=False)

        outputs, hidden = self.gru(packed, hidden)

        outputs, _ = nn.utils.rnn.pad_packed_sequence(outputs, batch_first=True)

        outputs = outputs[:, :, :self.hidden_size] + outputs[:, :, self.hidden_size:]
        hidden = hidden[0:1] + hidden[1:2]

        return outputs, hidden

In [7]:
class Attention(nn.Module):
    def __init__(self, hidden_size):
        super(Attention, self).__init__()
        self.hidden_size = hidden_size
        # We don't define self.softmax here to avoid confusion with dims

    def dot_scores(self, hidden: torch.Tensor, encoder_output):
        # hidden: (1, B, H) -> (B, H, 1)
        hidden = hidden.permute(1, 2, 0) 
        # (B, L, H) x (B, H, 1) -> (B, L, 1)
        return torch.bmm(encoder_output, hidden) 

    def forward(self, hidden, encoder_outputs):
        # 1. Calculate raw scores
        attention_scores = self.dot_scores(hidden, encoder_outputs) # (B, L, 1)
        
        # 2. Transpose to (B, 1, L) so we have all scores for the sequence in the last dim
        attention_scores = attention_scores.transpose(1, 2) 

        # 3. Apply Softmax CORRECTLY across the Length dimension
        attention_weights = torch.softmax(attention_scores, dim=2) 

        # 4. Weighted sum
        context_vector = torch.bmm(attention_weights, encoder_outputs) # (B, 1, H)

        return context_vector.permute(1, 0, 2)


In [8]:
class Decoder(nn.Module):
    def __init__(self, embedding, hidden_size, output_size, layers=1, dropout=0) -> None:
        super(Decoder, self).__init__()

        self.hidden_size = hidden_size

        self.embedding = embedding
        self.embedding_dropout = nn.Dropout(dropout)

        self.gru = nn.GRU(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=layers,
            dropout = (0 if layers == 1 else dropout),
            bidirectional = False, 
            batch_first=True
        )
        self.attention = Attention(hidden_size)
        self.concat = nn.Linear(hidden_size * 2, hidden_size)
        self.out = nn.Linear(hidden_size, output_size)


    def forward(self, current_input, last_hidden: torch.Tensor, encoder_states):
        
        embedded = self.embedding(current_input)
        embedded = self.embedding_dropout(embedded)

        gru_output, hidden = self.gru(embedded, last_hidden) # dims of output: (1, B, H)

        context_vector = self.attention(hidden, encoder_states).transpose(0, 1) # dims: (B, 1, H)

        context_vector = context_vector.squeeze(1)
        gru_output = gru_output.squeeze(1)

        # concatenating the input embedding and the context vector and then passing them through LC 
        concat_input = torch.concat([context_vector, gru_output], dim=1)
        concat_outputs = torch.tanh(self.concat(concat_input))

        output = self.out(concat_outputs)

        return output, hidden


In [9]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super(Seq2Seq, self).__init__()

        self.encoder = encoder
        self.decoder = decoder 
        self.device = device


    def forward(self, input_seq: torch.Tensor, input_lens, target_seq: torch.Tensor, criterion, teacher_forcing_ratio=0.5):
        

        batch_size = input_seq.shape[0]
        max_length = input_seq.shape[1]
        vocab_size = self.decoder.out.out_features

        
        encoder_bank, hidden = self.encoder(input_seq, input_lens)
        
        current_input = input_seq[:, 0].unsqueeze(1) # (B) -> (B, 1)

        loss = 0

        for t in range(1, max_length):
            
            output, hidden = self.decoder(current_input, hidden, encoder_bank)

            loss += criterion(output, target_seq[:, t])

            teacher_force = random.random() < teacher_forcing_ratio
            predicted_word = output.argmax(1)

            # next input
            current_input = target_seq[:, t].unsqueeze(1) if teacher_force else predicted_word.unsqueeze(1)

        return loss/max_length

In [10]:
# train function

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EMB_DIM = NUM_UNIQUE_WORDS
DROPOUT = 0.5
EPOCHS = 100

criterion = nn.CrossEntropyLoss(ignore_index=0) 
embedding = nn.Embedding(NUM_UNIQUE_WORDS, 128)
encoder = Encoder(hidden_size=128, embedding=embedding, layers=1).to(DEVICE)
decoder = Decoder(hidden_size=128, output_size=NUM_UNIQUE_WORDS, embedding=embedding, layers=1).to(DEVICE)
model = Seq2Seq(encoder=encoder, decoder=decoder, device=DEVICE).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# training loop. 

def train_batch(input_seq, input_lens, target_seq, target_lens):
    model.train()
    optimizer.zero_grad()

    input_seq = input_seq.to(DEVICE)
    target_seq = target_seq.to(DEVICE)

    loss = model(input_seq, input_lens, target_seq, target_lens, criterion)

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1)
    optimizer.step()

    return loss.item()


In [ ]:
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

class MyTextDataset(Dataset):
    def __init__(self, data):
        self.data = data
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx]

def my_collate(batch):
    inputs = [torch.tensor(item[0]) for item in batch]
    targets = [torch.tensor(item[1]) for item in batch]
    input_lens = torch.tensor([len(seq) for seq in inputs])
    
    inputs_padded = pad_sequence(inputs, batch_first=True, padding_value=0) # Uses <PAD> 0
    targets_padded = pad_sequence(targets, batch_first=True, padding_value=0)
    
    return inputs_padded, input_lens, targets_padded

train_data = []
for i in range(len(dataset) - 1):
    input_sent = dataset[i]
    target_sent = dataset[i+1]
    
    # Optional: Filter out super short lines (like "No") if you want smarter replies
    # But for now, let's keep everything
    train_data.append((input_sent, target_sent))

train_data.sort(key=lambda x: len(x[0]))

train_dataset = MyTextDataset(train_data)



# Create the Loader
train_loader = DataLoader(
    train_dataset, 
    batch_size=128, 
    shuffle=False, 
    collate_fn=my_collate,
    num_workers=4,
    pin_memory=True
)

loss = 0

print("Starting Training...")
for epoch in range(EPOCHS):
    epoch_loss = 0
    
    for src, src_len, trg in train_loader:
        # Pass only what the modified forward needs
        # Note: We don't pass target_lens anymore
        
        model.train()
        optimizer.zero_grad()

        src = src.to(DEVICE)
        trg = trg.to(DEVICE)

        loss = model(src, src_len, trg, criterion)

        # Flatten for Loss: (Batch * Len, Vocab) vs (Batch * Len)
        # Ignore Index 0 (Padding) so we don't learn from empty space

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1)
        optimizer.step()
        
        epoch_loss += loss.item()
        
    print(f"Epoch {epoch+1} | Loss: {epoch_loss / len(train_loader):.4f}")

Starting Training...


IndexError: index 43 is out of bounds for dimension 1 with size 43